In [ ]:
import requests
import json
import scanpy as sc
import os
import tqdm

from perturbgen.configs import ROOT

In [ ]:
# set wd
os.chdir(ROOT)
print('Current working directory:', ROOT)

Current working directory: /lustre/scratch126/cellgen/lotfollahi/kl11


In [ ]:
adata_clustered = sc.read_h5ad('T_perturb/res/hspc/perturbation_5k_res/summary_plots/20250723_leiden_pert_cls_annotated.h5ad')

In [ ]:
res_dir = '/lustre/scratch126/cellgen/lotfollahi/kl11/T_perturb/res/hspc/OT'
# create directory if it does not exist
if not os.path.exists(res_dir):
    os.makedirs(res_dir)

In [ ]:
QUERY = """
query TargetAssociationsQuery(
  $id: String!
  $index: Int!
  $size: Int!
  $sortBy: String!
  $enableIndirect: Boolean!
  $datasources: [DatasourceSettingsInput!]
  $rowsFilter: [String!]
  $facetFilters: [String!]
  $entitySearch: String!
) {
  target(ensemblId: $id) {
    id
    approvedSymbol
    associatedDiseases(
      page: { index: $index, size: $size }
      orderByScore: $sortBy
      enableIndirect: $enableIndirect
      datasources: $datasources
      Bs: $rowsFilter
      facetFilters: $facetFilters
      BFilter: $entitySearch
    ) {
      count
      rows {
        disease {
          id
          name
          ancestors
          therapeuticAreas {
            id
            name
          }
        }
        score
        datasourceScores {
          componentId: id
          score
        }
      }
    }
  }
}
"""

In [ ]:
VARIABLES = {
  "id": "ENSG00000061918",
  "index": 0,
  "size": 1000,
  "sortBy": "score",
  "enableIndirect": False,
  "datasources": [
    {"id": "gwas_credible_sets", "weight": 0.5, "propagate": True, "required": True},
    {"id": "gene_burden", "weight": 1, "propagate": True, "required": True},
    {"id": "eva", "weight": 0.5, "propagate": True, "required": True},
    {"id": "genomics_england", "weight": 1, "propagate": True, "required": True},
    {"id": "chembl", "weight": 1, "propagate": True, "required": True},
    {"id": "crispr_screen", "weight": 1, "propagate": True, "required": True},
    {"id": "crispr", "weight": 1, "propagate": True, "required": True},

  ],
  "entity": "target",
  "entitySearch": "",
  "rowsFilter": None,   # or a list of strings, e.g. ["someFilter"]
  "facetFilters": None  # or a list of strings, e.g. ["facet:value"]
}


In [ ]:
# Set base URL of GraphQL API endpoint
base_url = "https://api.platform.opentargets.org/api/v4/graphql"


In [ ]:
query_results = {}
failed = {}

for gene in tqdm.tqdm(adata_clustered.obs["ensembl_id"].unique().tolist()):
    VARIABLES["id"] = gene
    try:
        r = requests.post(base_url, json={"query": QUERY, "variables": VARIABLES}, timeout=30)

        if r.status_code != 200:
            failed[gene] = {"status_code": r.status_code, "body": r.text[:300]}
            continue

        query_results[gene] = r.json()

    except Exception as e:
        failed[gene] = {"error": repr(e)}

  0%|          | 0/3107 [00:00<?, ?it/s]

 59%|█████▉    | 1838/3107 [02:19<01:26, 14.63it/s]

In [ ]:
# create a pandas dataframe from query_results
import pandas as pd

dfs = []
for result in query_results:
    rows = query_results[result]['data']
    if rows['target'] is None:
        print("No data for ", result)
        # empty dataframe
        df = pd.DataFrame()
        df['ensembl_id'] = [result]
        dfs.append(df)
    else:
        df = pd.json_normalize(
            rows["target"]["associatedDiseases"]["rows"],
            record_path=["datasourceScores"],
            meta=[
                ["disease", "id"],
                ["disease", "name"],
                ["disease", "ancestors"],
                ["disease", "therapeuticAreas"],
                # ["disease", "therapeuticAreas", "name"],
                "score"
            ],  # columns to retain
            record_prefix="ds_",
            meta_prefix="assoc_",
            errors="ignore"
            )
        df['ensembl_id'] = result
        dfs.append(df)
final_df = pd.concat(dfs, ignore_index=True)
# nice column names
final_df = final_df.rename(columns={
    "ds_componentId": "datasource",
    "ds_score": "datasource_score",
    "assoc_score": "association_score",
    "assoc_disease.id": "disease_id",
    "assoc_disease.ancestors": "disease_ancestors",
    "assoc_disease.name": "disease_name",
    "assoc_disease.therapeuticAreas": "therapeutic_areas",
})


No data for  ENSG00000273748


In [ ]:
# transform list to string as list is unhashable for pivot table
final_df['disease_ancestors'] = final_df['disease_ancestors'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')

final_df = final_df.explode("therapeutic_areas", ignore_index=True)
final_df["therapeutic_area_id"] = final_df["therapeutic_areas"].apply(
    lambda x: x.get("id") if isinstance(x, dict) else None      
)
final_df["therapeutic_area_name"] = final_df["therapeutic_areas"].apply(
    lambda x: x.get("name") if isinstance(x, dict) else None
)
final_df.drop(columns=["therapeutic_areas"], inplace=True)

In [ ]:
final_df = final_df[final_df['datasource_score'].notna()]
final_df = final_df[final_df['datasource'].isin(['gwas_credible_sets', 'gene_burden', 'eva', 'genomics_england', 'chembl'])]

In [ ]:
# map gene name to ensembl_ids
ensemblid_to_genes = dict(zip(adata_clustered.obs['ensembl_id'], adata_clustered.obs['genes']))
final_df['genes'] = final_df['ensembl_id'].map(ensemblid_to_genes)
ensemblid_to_cluster = dict(zip(adata_clustered.obs['ensembl_id'], adata_clustered.obs['GP_name']))
final_df['cluster'] = final_df['ensembl_id'].map(ensemblid_to_cluster)

In [ ]:
final_df

,datasource,datasource_score,disease_id,disease_name,disease_ancestors,association_score,ensembl_id,therapeutic_area_id,therapeutic_area_name,genes,cluster
0,gwas_credible_sets,0.742735,EFO_0009188,Red cell distribution width,"EFO_0004509, EFO_0001444, EFO_0004503, EFO_000...",0.225766,ENSG00000032219,EFO_0001444,measurement,ARID4A,Activated antigen-presenting myeloid cells
1,gwas_credible_sets,0.711804,EFO_0004528,mean corpuscular hemoglobin concentration,"EFO_0001444, EFO_0004306, EFO_0004509, EFO_000...",0.216364,ENSG00000032219,EFO_0001444,measurement,ARID4A,Activated antigen-presenting myeloid cells
2,gwas_credible_sets,0.606964,EFO_0004530,triglyceride measurement,"EFO_0004747, EFO_0005105, PATO_0000070, EFO_00...",0.184496,ENSG00000032219,EFO_0001444,measurement,ARID4A,Activated antigen-presenting myeloid cells
3,gwas_credible_sets,0.591136,EFO_0004527,mean corpuscular hemoglobin,"EFO_0004306, EFO_0004503, EFO_0001444",0.179685,ENSG00000032219,EFO_0001444,measurement,ARID4A,Activated antigen-presenting myeloid cells
4,gwas_credible_sets,0.587169,OBA_0003460,erythrocyte volume,"OBA_2045276, EFO_0001444",0.178479,ENSG00000032219,EFO_0001444,measurement,ARID4A,Activated antigen-presenting myeloid cells
...,...,...,...,...,...,...,...,...,...,...,...
245391,gwas_credible_sets,0.075050,EFO_0009514,upper extremity fracture,"OTAR_0000006, EFO_0002461, EFO_0009676, EFO_00...",0.022813,ENSG00000128833,OTAR_0000009,"injury, poisoning or other complication",MYO5C,TNF and lymphoid-biased haematopoiesis
245392,gwas_credible_sets,0.074120,MONDO_0002076,pneumothorax,"EFO_0000684, MONDO_0002037, EFO_0009433, OTAR_...",0.02253,ENSG00000128833,OTAR_0000010,respiratory or thoracic disease,MYO5C,TNF and lymphoid-biased haematopoiesis
245393,gwas_credible_sets,0.074120,EFO_0009680,pleural empyema,"OTAR_0000010, EFO_0000684, MONDO_0002037, EFO_...",0.02253,ENSG00000128833,OTAR_0000010,respiratory or thoracic disease,MYO5C,TNF and lymphoid-biased haematopoiesis
245394,gwas_credible_sets,0.063226,EFO_0009012,Polyarteritis Nodosa,"EFO_0009011, MONDO_0000473, EFO_0000319, EFO_0...",0.019219,ENSG00000128833,EFO_0000319,cardiovascular disease,MYO5C,TNF and lymphoid-biased haematopoiesis


In [ ]:
# create a wider dataframe with pivot table using datasource as columns
final_df = final_df.pivot_table(index=['ensembl_id', 'genes', 'cluster', 'disease_id', 'disease_name', 'therapeutic_area_name', 'therapeutic_area_id', 'association_score', 'disease_ancestors'],
                                columns='datasource',
                                values='datasource_score').reset_index()

In [ ]:
# filter out columns where chembl eva gene_burden genomics_england are all Nan and gwas_credible_sets < 0.5

final_df = final_df[
    (final_df['gwas_credible_sets'] >= 0.5 # filter locus_to_gene score
    )  | 
    (
        final_df['eva'] >= 0.5  # 0.5 at least risk factor https://platform-docs.opentargets.org/evidence#clinvar
    ) |
    (
        final_df['gene_burden'] >= 0
    ) |
    (
        final_df['genomics_england'] >= 0
    )
]

In [ ]:
final_df

datasource,ensembl_id,genes,cluster,disease_id,disease_name,therapeutic_area_name,therapeutic_area_id,association_score,disease_ancestors,chembl,eva,gene_burden,genomics_england,gwas_credible_sets
0,ENSG00000000971,CFH,Hemoglobin synthesis,EFO_0000719,temporal measurement,measurement,EFO_0001444,0.131209,EFO_0001444,NaN,NaN,NaN,NaN,0.431658
1,ENSG00000000971,CFH,Hemoglobin synthesis,EFO_0001365,age-related macular degeneration,disorder of visual system,MONDO_0024458,0.762846,"EFO_0009606, MONDO_0004580, EFO_0003966, EFO_0...",NaN,NaN,0.151983,0.607931,0.903470
2,ENSG00000000971,CFH,Hemoglobin synthesis,EFO_0001365,age-related macular degeneration,"genetic, familial or congenital disease",OTAR_0000018,0.762846,"EFO_0009606, MONDO_0004580, EFO_0003966, EFO_0...",NaN,NaN,0.151983,0.607931,0.903470
3,ENSG00000000971,CFH,Hemoglobin synthesis,EFO_0001365,age-related macular degeneration,nervous system disease,EFO_0000618,0.762846,"EFO_0009606, MONDO_0004580, EFO_0003966, EFO_0...",NaN,NaN,0.151983,0.607931,0.903470
4,ENSG00000000971,CFH,Hemoglobin synthesis,EFO_0001365,age-related macular degeneration,psychiatric disorder,MONDO_0002025,0.762846,"EFO_0009606, MONDO_0004580, EFO_0003966, EFO_0...",NaN,NaN,0.151983,0.607931,0.903470
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185729,ENSG00000278129,ZNF8,Core transcription control,EFO_0011015,educational attainment,measurement,EFO_0001444,0.012565,EFO_0001444,NaN,NaN,NaN,NaN,0.041338
185730,ENSG00000284691,AC073111.4,Developmental transcriptional regulation,EFO_0000694,severe acute respiratory syndrome,infectious disease,EFO_0005741,0.072378,"MONDO_0020753, EFO_0007223, OTAR_0000010, EFO_...",NaN,NaN,NaN,NaN,0.238112
185731,ENSG00000284691,AC073111.4,Developmental transcriptional regulation,EFO_0000694,severe acute respiratory syndrome,respiratory or thoracic disease,OTAR_0000010,0.072378,"MONDO_0020753, EFO_0007223, OTAR_0000010, EFO_...",NaN,NaN,NaN,NaN,0.238112
185732,ENSG00000284691,AC073111.4,Developmental transcriptional regulation,MONDO_0100096,COVID-19,infectious disease,EFO_0005741,0.072378,"MONDO_0020753, EFO_0007223, MONDO_0100329, EFO...",NaN,NaN,NaN,NaN,0.238112


In [ ]:
# save 
final_df.to_csv(f'{res_dir}/OT.csv')